# Domain Analysis with pySpectrum

A `Domain` is a contiguous slice of a `Spectrum` — the fundamental object for localised analysis. Once a region of interest is extracted, pySpectrum provides:

- **Background estimation** — erf step or linear interpolation between domain edges
- **Peak centre and FWHM** — weighted centroid and half-maximum crossing
- **Statistical moments** — centroid, variance, skewness
- **SNR-based domain finding** — automatic detection of all significant peak regions

## Workflow
1. Load and calibrate the spectrum
2. Automatically find peak domains via SNR convolution
3. Analyse a single domain (background, centre, FWHM)
4. Inspect moment estimators and domain-level properties

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyspectrum.core import Spectrum
from pyspectrum.calibration import AxisCalibration, ResolutionCalibration
from pyspectrum.calibration.models.hpge_fwhm_model import StandardHPGeFWHMModel
from pyspectrum.io import TimeChannelParser
from pyspectrum.identification import Convolution
from pyspectrum.identification.snr import SNRFinder
from pyspectrum.identification.kernels.mexican_hat import gaussian_2_dev
from pyspectrum.domain_analysis.single_peak import center_estimator, fwhm_estimator
from pyspectrum.domain_analysis.background import domain_erf_background
from pyspectrum.domain_analysis.moment import centroid

## 1. Load and calibrate spectrum

> **Adapt:** replace `path` and the calibration parameters with values for your own detector and source.

In [ ]:
# ── Adapt to your detector ────────────────────────────────────────────────────
path                   = '../Library/Doppler_broadening_Spectrum.txt'
energy_calib_poly      = np.poly1d([0.0408976444, 0.0822321508])  # channel → keV
energy_resolution_fwhm = 1.05   # FWHM at 511 keV [keV]
num_of_channels        = 16384
# ─────────────────────────────────────────────────────────────────────────────

energy_calib  = AxisCalibration(func=energy_calib_poly, name="energy")
estimated_FWHM = StandardHPGeFWHMModel().generator((0, energy_resolution_fwhm / 511**0.5, 0))
res_calib     = ResolutionCalibration(func=estimated_FWHM)

spectrum = TimeChannelParser.from_file(
    path, axis_calib=energy_calib, resolution_calib=res_calib,
    num_of_channels=num_of_channels, chunk_size=10_000,
    sep=' ', skiprows=5, names=['time', 'channel', 'flag'], usecols=[0, 1, 2],
)

In [ ]:
spectrum.data.plot(yscale='log')
plt.title('Doppler broadening spectrum')
plt.xlabel('Energy [keV]')
plt.ylabel('Counts')
plt.grid(True, which='both')
plt.xlim([300, 700])
plt.tight_layout()
plt.show()

## 2. Find peak domains via SNR convolution

`SNRFinder` convolves the spectrum with a zero-area Gaussian second-derivative kernel (Mexican hat). Where the convolution response exceeds the signal threshold, the algorithm expands a domain outward until the SNR drops below the background threshold for a full persistence window.

> **Adapt:** `n_sigma_signal_threshold` sets the detection sensitivity; `n_sigma_bg_threshold` and `persistence_factor` control domain edge placement. Higher thresholds reduce false positives but may miss weak peaks.

In [ ]:
# ── Adapt detection thresholds ────────────────────────────────────────────────
n_sigma_signal     = 4.0   # detection threshold
n_sigma_background = 2.0   # domain edge threshold
persistence        = 0.5   # fraction of FWHM for persistence window
# ─────────────────────────────────────────────────────────────────────────────

conv   = Convolution(resolution=res_calib.apply, kernel=gaussian_2_dev, window_fwhm=3)
finder = SNRFinder(convolution=conv,
                   n_sigma_signal_threshold=n_sigma_signal,
                   n_sigma_bg_threshold=n_sigma_background,
                   persistence_factor=persistence)

pyspecrtum use SNR method to find meaningfull domains. \
the convolution is not standard numpy function because the width depends on the axis 

In [6]:
conv = Convolution(resolution=res_calib.apply, kernel=gaussian_2_dev, window_fwhm=3)
finder = SNRFinder(convolution=conv, n_sigma_signal_threshold=4, n_sigma_bg_threshold=2.0, persistence_factor=0.5)

## 3. Extract and analyse a single domain

`finder.domain(spectrum, axis_value)` returns the domain surrounding the peak nearest to the requested energy. Once extracted, the domain can be background-subtracted and its peak properties estimated with dedicated estimators.

In [ ]:
annihilation_domain = finder.domain(spectrum, 511.0)

annihilation_domain.data.plot()
plt.title('511 keV annihilation peak domain')
plt.xlabel('Energy [keV]')
plt.ylabel('Counts')
plt.grid(True, which='both')
plt.tight_layout()
plt.show()

print(f"Domain: {spectrum.axis[annihilation_domain.start]:.2f} – "
      f"{spectrum.axis[annihilation_domain.stop]:.2f} keV")

### 3a. Background subtraction and peak properties

`domain_erf_background` estimates the local Compton step background using a prominence-weighted erf model. `subtract_background` returns a **new** domain with the background lazily attached — the original domain is unchanged.

`center_estimator` computes the peak centre as a Poisson-weighted centroid within the FWHM region, with uncertainty propagated from `counts_err` if available. `fwhm_estimator` uses half-maximum crossings with parabolic interpolation.

In [ ]:
estimated_background = domain_erf_background(annihilation_domain)
subtracted_domain    = annihilation_domain.subtract_background(estimated_background)

center = center_estimator(annihilation_domain)
fwhm   = fwhm_estimator(annihilation_domain)
print(f"Centre: {center} keV")
print(f"FWHM:   {fwhm:.4f} keV")

In [23]:
# You can analyze the peaks fwhm, center, and background
print(f" center {center_estimator(annihilation_domain)}, and fwhm -{fwhm_estimator(annihilation_domain)}")
 
estimated_background = domain_erf_background(annihilation_domain)
# subtraction returns subtracted domain
# Note the pySpectrum handles the error for you
subtracted_domain = annihilation_domain.subtract_background(estimated_background)

 center 511.1984+/-0.0009, and fwhm -2.6993062945404063


In [ ]:
annihilation_domain.data.plot.step(yscale='log', label='original')
subtracted_domain.data.plot.step(label='background subtracted')
plt.step(annihilation_domain.data.energy, estimated_background, label='background', ls='--')
plt.title('511 keV annihilation peak — background subtraction')
plt.xlabel('Energy [keV]')
plt.ylabel('Counts')
plt.ylim([1, annihilation_domain.max()])
plt.legend()
plt.grid(True, which='both')
plt.tight_layout()
plt.show()

## 4. Full-spectrum domain identification and moment analysis

`finder.find(spectrum)` returns all significant peak domains across the full spectrum. Each domain supports arithmetic and xarray operations directly, and statistical moment estimators (centroid, FWHM) can be applied to any domain.

In [ ]:
conv2   = Convolution(resolution=res_calib.apply, kernel=gaussian_2_dev, window_fwhm=4)
finder2 = SNRFinder(convolution=conv2, n_sigma_signal_threshold=4,
                    n_sigma_bg_threshold=2, persistence_factor=0.5)
domains = finder2.find(spectrum)
print(f"Found {len(domains)} peak domains")

In [ ]:
spectrum.data.plot(yscale='log', color='steelblue', label='spectrum')
for domain in domains:
    domain.data.plot(color='tomato')   # highlight detected domains in red

plt.xlim([300, 700])
plt.title('Detected peak domains — Doppler broadening spectrum')
plt.xlabel('Energy [keV]')
plt.ylabel('Counts')
plt.grid(True, which='both')
plt.tight_layout()
plt.show()

### 4a. Moment and resolution estimators on a detected domain

`centroid` computes the first moment (centre of mass). `center_estimator` uses a more robust Poisson-weighted centroid within the FWHM region. `domain.local_resolution` reads the detector resolution calibration at the domain centre.

In [32]:
from pyspectrum.domain_analysis.moment import centroid

In [ ]:
domain = domains[-1]   # last detected domain (highest energy in the window)

print(f"Centroid (moment):   {centroid(domain):.4f} keV")
print(f"Centre (estimator):  {center_estimator(domain)}")
print(f"FWHM from calib:     {domain.local_resolution:.4f} keV")
print(f"FWHM from estimator: {fwhm_estimator(domain):.4f} keV")

In [39]:
# resolution can be taken from the calibration or from estimators
fwhm = domains[-1].local_resolution
print(f"fwhm from calibration - {fwhm:.2f}, and from direct estimation {fwhm_estimator(domain):.2f}")

fwhm from calibration - 1.15, and from direct estimation 1.24


In [ ]:
fwhm       = domain.local_resolution
mean_energy = centroid(domain)
half_max    = domain.data.max() / 2

domain.data.plot(marker='.', label='domain')
plt.hlines(y=half_max,
           xmin=mean_energy - fwhm / 2,
           xmax=mean_energy + fwhm / 2,
           colors='r', lw=2, label=f'FWHM = {fwhm:.2f} keV')
plt.vlines(x=[mean_energy - fwhm/2, mean_energy + fwhm/2],
           ymin=0, ymax=half_max, colors='r', ls='--', alpha=0.4)
plt.title(f'Detected domain — centroid {mean_energy:.2f} keV')
plt.xlabel('Energy [keV]')
plt.ylabel('Counts')
plt.legend()
plt.grid(True, which='both')
plt.tight_layout()
plt.show()